# N3 — Metaheurísticas (PSO) · Telemetry Heart AI

Evidencia de la rúbrica N3 para el optimizador **PSO** (única técnica SI1):
priorización clínica de triaje en 3 niveles (BAJA/MEDIA/ALTA) optimizando
**7 pesos + 2 umbrales**.

- **Métrica #1**: accuracy (baseline → optimizado)
- **Métrica #2**: critical_recall (recall de la clase tope ALTA) y overtriage_rate (seguridad clínica)
- **Visualización**: curva de convergencia + matriz de confusión 3×3

El scoring (`Σ(f·w)/Σ|w|`) y la normalización de features son **idénticos** a los de
producción (`TriagePriorityService` / `build_feature_bundle`), por lo que el
antes/después refleja el modelo real servido.

In [1]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# Ejecutar desde la raíz del microservicio
if Path.cwd().name == 'notebooks':
    os.chdir('..')

from app.tools.pso_tools import _load_dataset
from app.services.metrics_service import MetricsService
from app.services.optimizers.pso import _weighted_scores, _classify_priority

X, y = _load_dataset(Path('app/data/synthetic_cases.csv'))
weights_json = json.loads(Path('app/data/triage_priority_weights.json').read_text(encoding='utf-8'))
curve_json = json.loads(Path('app/data/convergence_curve.json').read_text(encoding='utf-8'))

opt_w = np.array(weights_json['weights'])
opt_t = weights_json['thresholds']
print('X', X.shape, '| clases y:', np.bincount(y))
print('version:', weights_json['version'])

C:\Users\HALO\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


X (500, 7) | clases y: [250 120 130]
version: pso-2026-06-demo


In [2]:
# Baseline = pesos uniformes (1/7) + umbrales por defecto; Optimizado = salida PSO.
ms = MetricsService()
base_w = np.full(7, 1.0 / 7.0)
base_t = [0.40, 0.70]

cmp = ms.compare(X, y, base_w, base_t, opt_w, opt_t)
rows = ['accuracy', 'f1_score', 'critical_recall', 'overtriage_rate', 'ordinal_error', 'fitness']
table = pd.DataFrame({
    'Métrica': rows,
    'Baseline': [round(cmp['baseline'][k], 4) for k in rows],
    'Optimizado (PSO)': [round(cmp['optimized'][k], 4) for k in rows],
    'Δ': [round(cmp['improvement'][k], 4) for k in rows],
})
print('N3 — Comparación baseline vs PSO')
print(table.to_string(index=False))
table

N3 — Comparación baseline vs PSO
        Métrica  Baseline  Optimizado (PSO)       Δ
       accuracy    0.6920            0.9900  0.2980
       f1_score    0.5856            0.9879  0.4023
critical_recall    0.2615            1.0000  0.7385
overtriage_rate    0.0000            0.0081  0.0081
  ordinal_error    0.1540            0.0050 -0.1490
        fitness    2.3694            0.0091 -2.3603


,Métrica,Baseline,Optimizado (PSO),Δ
0,accuracy,0.6920,0.9900,0.2980
1,f1_score,0.5856,0.9879,0.4023
2,critical_recall,0.2615,1.0000,0.7385
3,overtriage_rate,0.0000,0.0081,0.0081
4,ordinal_error,0.1540,0.0050,-0.1490
5,fitness,2.3694,0.0091,-2.3603


In [3]:
# Curva de convergencia (fitness por iteración, a minimizar).
curve = curve_json['convergence_curve']
plt.figure(figsize=(8, 4))
plt.plot(range(1, len(curve) + 1), curve, marker='o', ms=3)
plt.title(f"Convergencia PSO ({curve_json['n_iterations']} iteraciones)")
plt.xlabel('Iteración')
plt.ylabel('Mejor fitness (gbest)')
plt.grid(alpha=0.3)
Path('app/data/charts').mkdir(parents=True, exist_ok=True)
plt.tight_layout()
plt.savefig('app/data/charts/n3_convergence.png', dpi=120)
plt.show()

C:\Users\HALO\AppData\Local\Temp\ipykernel_24244\1608300533.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [4]:
# Matriz de confusión 3x3 del modelo optimizado.
scores = _weighted_scores(np.clip(X, 0, 1), opt_w)
preds = np.array([_classify_priority(s, np.array(opt_t)) for s in scores])
labels = ['BAJA', 'MEDIA', 'ALTA']
cm = confusion_matrix(y, preds, labels=[0, 1, 2])

n = len(labels)
fig, ax = plt.subplots(figsize=(5.5, 5))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(n)); ax.set_xticklabels(labels)
ax.set_yticks(range(n)); ax.set_yticklabels(labels)
ax.set_xlabel('Predicho'); ax.set_ylabel('Real')
ax.set_title('Matriz de confusión — PSO optimizado')
for i in range(n):
    for j in range(n):
        ax.text(j, i, cm[i, j], ha='center', va='center',
                color='white' if cm[i, j] > cm.max() / 2 else 'black')
fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig('app/data/charts/n3_confusion.png', dpi=120)
plt.show()

C:\Users\HALO\AppData\Local\Temp\ipykernel_24244\894101171.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
